<a href="https://colab.research.google.com/github/FernandoJavierNegro/Prueba-2026/blob/main/PRUEBA2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/FernandoJavierNegro/Prueba-2026/blob/main/PRUEBA2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# EfficientDet-Lite1 para detectar botellas

Este cuaderno prepara un dataset YOLO en `.zip`, valida im?genes y etiquetas `.txt`, divide los datos en entrenamiento, validaci?n y prueba, entrena EfficientDet-Lite1 con TensorFlow Lite Model Maker y descarga un paquete listo para Android.

Importante: TensorFlow Lite Model Maker no funciona bien instalado directamente sobre el Python actual de Colab. Por eso este notebook crea un entorno Python 3.9 separado y ejecuta el entrenamiento como script dentro de ese entorno compatible.

Active GPU antes de empezar: `Entorno de ejecuci?n > Cambiar tipo de entorno de ejecuci?n > GPU`.

Formato esperado del ZIP:

```text
dataset/
??? images/
?   ??? foto1.jpg
?   ??? foto2.jpg
??? labels/
?   ??? foto1.txt
?   ??? foto2.txt
??? data.yaml
```

Tambi?n admite subcarpetas como `images/train`, `images/val`, `labels/train` y `labels/val`.

Cada `.txt` debe usar formato YOLO:

```text
class_id x_center y_center width height
```

Para una sola clase de botellas:

```text
0 0.5321 0.4812 0.2843 0.6115
```


## Celda 1.  Crear entorno compatible

Ejecute esta celda una sola vez por sesión de Colab. Puede tardar varios minutos.


In [4]:
%%bash
set -e

# ============================================================
# CREAR ENTORNO COMPATIBLE PARA TFLITE MODEL MAKER
# ============================================================

CONDA_DIR="/content/miniconda"
ENV_NAME="tflite_mm39"

if [ ! -d "$CONDA_DIR" ]; then
    echo "Instalando Miniconda..."
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-py39_23.3.1-0-Linux-x86_64.sh -O /content/miniconda.sh
    bash /content/miniconda.sh -b -f -p "$CONDA_DIR"
fi

source "$CONDA_DIR/etc/profile.d/conda.sh"

# Limpiar entornos fallidos de pruebas anteriores.
conda env remove -n tflite_mm311 -y >/dev/null 2>&1 || true
conda env remove -n tflite_mm310 -y >/dev/null 2>&1 || true

if ! conda env list | awk '{print $1}' | grep -qx "$ENV_NAME"; then
    echo "Creando entorno $ENV_NAME con Python 3.9..."
    conda create -y -n "$ENV_NAME" python=3.9
fi

conda activate "$ENV_NAME"

python -m pip install -q --upgrade pip==23.3.2 setuptools==65.5.1 wheel

# Versiones compatibles con TensorFlow Lite Model Maker 0.4.3 y scann 1.2.6.
python -m pip install -q --no-cache-dir     numpy==1.23.3     pandas==1.5.3     scipy==1.10.1     scikit-learn==1.2.2     protobuf==3.19.6     pillow==9.5.0     pyyaml==6.0.1     pycocotools==2.0.7 \
    matplotlib==3.4.3 \
    matplotlib-inline==0.1.7 \
    ipython==8.18.1 \
    packaging==20.9

python -m pip install -q --no-cache-dir tflite-model-maker==0.4.3

python - <<'PY'
from tflite_model_maker import model_spec, object_detector
from tflite_model_maker.config import QuantizationConfig
print("? TensorFlow Lite Model Maker funciona en el entorno Python 3.9.")
PY

echo " Celda 1 finalizada. Continúe con la celda 2."


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.5 MB/s eta 0:00:00
? TensorFlow Lite Model Maker funciona en el entorno Python 3.9.
 Celda 1 finalizada. Continúe con la celda 2.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflowjs 3.18.0 requires packaging~=20.9, but you have packaging 26.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wheel 0.47.0 requires packaging>=24.0, but you have packaging 20.9 which is incompatible.
2026-08-03 19:24:10.834547: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/lib64-nvidia
2026-08-03 19:24:10.834596: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
/content/miniconda/envs/tflite_mm39/lib

## Celda 2. Subir el ZIP del dataset
Formato esperado del ZIP:


```text
dataset/
???   images/
    ??? foto1.jpg  
    ??? foto2.jpg
??? labels/
    ??? foto1.txt
    ??? foto2.txt
??? data.yaml

Seleccionar esta carpeta para el drive

/content/drive/MyDrive/heladera_dataset_preparado/dataset_640_validas/images.zip


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# ============================================================
# SUBIR DATASET YOLO EN ZIP
# Opcion 1: desde la computadora
# Opcion 2: desde Google Drive con progreso
# ============================================================

from pathlib import Path
from google.colab import files, drive
import shutil
import time

def copiar_con_progreso(origen, destino, bloque_mb=8):
    origen = Path(origen)
    destino = Path(destino)

    total = origen.stat().st_size
    copiado = 0
    bloque = bloque_mb * 1024 * 1024

    with open(origen, "rb") as f_origen, open(destino, "wb") as f_destino:
        while True:
            datos = f_origen.read(bloque)
            if not datos:
                break

            f_destino.write(datos)
            copiado += len(datos)

            porcentaje = (copiado / total) * 100
            print(f"\rCargando desde Drive: {porcentaje:6.2f}%", end="")

    print("\nCarga finalizada.")


print("Seleccione origen del dataset ZIP:")
print("1 = Subir desde mi computadora")
print("2 = Usar archivo desde Google Drive")

opcion = input("Escriba 1 o 2: ").strip()

zip_estandar = Path("/content/dataset_botellas.zip")

if zip_estandar.exists():
    zip_estandar.unlink()

if opcion == "1":
    print("\nSeleccione el archivo ZIP que contiene imagenes y etiquetas YOLO .txt")
    archivos_subidos = files.upload()

    if not archivos_subidos:
        raise RuntimeError("No se selecciono ningun archivo.")

    ruta_zip = None
    for nombre_archivo in archivos_subidos:
        if nombre_archivo.lower().endswith(".zip"):
            ruta_zip = Path("/content") / nombre_archivo
            break

    if ruta_zip is None:
        raise ValueError("Debe seleccionar un archivo .zip")

    shutil.move(str(ruta_zip), str(zip_estandar))

elif opcion == "2":
    print("\nMontando Google Drive...")
    drive.mount("/content/drive")

    print("\nPegue la ruta completa del ZIP en Drive.")
    print("Ejemplo:")
    print("/content/drive/MyDrive/datasets/images.zip")

    ruta_drive = Path(input("Ruta del ZIP en Drive: ").strip().strip('"').strip("'"))

    if not ruta_drive.exists():
        raise FileNotFoundError(f"No existe el archivo: {ruta_drive}")

    if ruta_drive.suffix.lower() != ".zip":
        raise ValueError("El archivo seleccionado debe ser .zip")

    tamaño_mb = ruta_drive.stat().st_size / (1024 * 1024)

    print("\nArchivo encontrado en Google Drive:")
    print(f"Ruta: {ruta_drive}")
    print(f"Tamaño: {tamaño_mb:.2f} MB")
    print("Iniciando carga hacia Colab...")

    copiar_con_progreso(ruta_drive, zip_estandar)

else:
    raise ValueError("Opcion invalida. Debe escribir 1 o 2.")

print(f"\nDataset cargado correctamente: {zip_estandar}")
print(f"Tamaño final: {zip_estandar.stat().st_size / (1024 * 1024):.2f} MB")

Seleccione origen del dataset ZIP:
1 = Subir desde mi computadora
2 = Usar archivo desde Google Drive
Escriba 1 o 2: 2

Montando Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Pegue la ruta completa del ZIP en Drive.
Ejemplo:
/content/drive/MyDrive/datasets/images.zip
Ruta del ZIP en Drive: /content/drive/MyDrive/heladera_dataset_preparado/dataset_640_validas/images.zip

Archivo encontrado en Google Drive:
Ruta: /content/drive/MyDrive/heladera_dataset_preparado/dataset_640_validas/images.zip
Tamaño: 17.32 MB
Iniciando carga hacia Colab...
Cargando desde Drive: 100.00%
Carga finalizada.

Dataset cargado correctamente: /content/dataset_botellas.zip
Tamaño final: 17.32 MB


In [7]:
# ============================================================
# REPARAR data.yaml DENTRO DEL ZIP
# ============================================================

from pathlib import Path
import zipfile
import shutil

zip_path = Path("/content/dataset_botellas.zip")
tmp_dir = Path("/content/dataset_yaml_reparado")

if not zip_path.exists():
    raise FileNotFoundError("No existe /content/dataset_botellas.zip")

if tmp_dir.exists():
    shutil.rmtree(tmp_dir)

tmp_dir.mkdir(parents=True, exist_ok=True)

print("Extrayendo ZIP...")
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(tmp_dir)

yaml_files = list(tmp_dir.rglob("data.yaml")) + list(tmp_dir.rglob("*.yml"))

if not yaml_files:
    raise RuntimeError("No se encontro data.yaml dentro del ZIP.")

yaml_path = yaml_files[0]

print("data.yaml encontrado:")
print(yaml_path)

yaml_corregido = """path: .
train: images
val: images
test: images
nc: 12
names:
- coca cola
- sprite
- agua
- jugo de naranja
- pepsi
- class_5
- class_6
- class_7
- class_8
- class_9
- class_10
- class_11
"""

yaml_path.write_text(yaml_corregido, encoding="utf-8")

print("\ndata.yaml corregido:")
print(yaml_path.read_text(encoding="utf-8"))

backup = Path("/content/dataset_botellas_backup_yaml_malo.zip")
if backup.exists():
    backup.unlink()

shutil.move(str(zip_path), str(backup))

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for archivo in tmp_dir.rglob("*"):
        if archivo.is_file():
            z.write(archivo, archivo.relative_to(tmp_dir))

print("\nZIP reparado correctamente:")
print(zip_path)
print("Backup del ZIP anterior:")
print(backup)

Extrayendo ZIP...
data.yaml encontrado:
/content/dataset_yaml_reparado/data.yaml

data.yaml corregido:
path: .
train: images
val: images
test: images
nc: 12
names:
- coca cola
- sprite
- agua
- jugo de naranja
- pepsi
- class_5
- class_6
- class_7
- class_8
- class_9
- class_10
- class_11


ZIP reparado correctamente:
/content/dataset_botellas.zip
Backup del ZIP anterior:
/content/dataset_botellas_backup_yaml_malo.zip


## Celda 3.  Crear script de entrenamiento


In [8]:
# ============================================================
# CELDA 3 - CREAR SCRIPT DE ENTRENAMIENTO MULTICLASE
# Lee TODAS las clases declaradas en data.yaml
# ============================================================

from pathlib import Path
script_path = Path("/content/train_efficientdet_lite1.py")
codigo = r'''

import argparse
import json
import random
import shutil
import zipfile
from pathlib import Path
from collections import Counter

import pandas as pd
import yaml
from PIL import Image, UnidentifiedImageError

from tflite_model_maker import model_spec, object_detector
from tflite_model_maker.config import ExportFormat, QuantizationConfig


BASE_DIR = Path("/content/efficientdet_lite1_botellas")
DATASET_EXTRAIDO_DIR = BASE_DIR / "dataset_extraido"
DATASET_PREPARADO_DIR = BASE_DIR / "dataset_preparado"
RESULTADOS_DIR = BASE_DIR / "resultados"
EXPORT_DIR = BASE_DIR / "modelo_exportado"

CSV_PATH = DATASET_PREPARADO_DIR / "annotations_model_maker.csv"
LABELS_PATH = DATASET_PREPARADO_DIR / "labels.txt"

SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.20
TEST_RATIO = 0.10

EPOCHS = 40
BATCH_SIZE = 4


def limpiar_carpetas():
    if BASE_DIR.exists():
        shutil.rmtree(BASE_DIR)

    DATASET_EXTRAIDO_DIR.mkdir(parents=True, exist_ok=True)
    DATASET_PREPARADO_DIR.mkdir(parents=True, exist_ok=True)
    RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)


def extraer_zip(zip_path):
    print(f"Descomprimiendo dataset: {zip_path}", flush=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(DATASET_EXTRAIDO_DIR)

    print(f"Dataset extraido en: {DATASET_EXTRAIDO_DIR}", flush=True)


def encontrar_data_yaml():
    yamls = list(DATASET_EXTRAIDO_DIR.rglob("data.yaml")) + list(DATASET_EXTRAIDO_DIR.rglob("*.yml"))

    if not yamls:
        raise RuntimeError("No se encontro data.yaml dentro del ZIP.")

    return yamls[0]


def cargar_clases_desde_yaml(yaml_path):
    data = yaml.safe_load(yaml_path.read_text(encoding="utf-8"))

    if not isinstance(data, dict):
        raise RuntimeError("data.yaml no tiene formato valido.")

    names = data.get("names")
    nc = data.get("nc")

    if names is None:
        raise RuntimeError("data.yaml no tiene campo names.")

    if isinstance(names, list):
        class_names = {i: str(nombre) for i, nombre in enumerate(names)}
    elif isinstance(names, dict):
        class_names = {int(k): str(v) for k, v in names.items()}
    else:
        raise RuntimeError("El campo names debe ser lista o diccionario.")

    if nc is not None and int(nc) != len(class_names):
        print(
            f"[ADVERTENCIA] data.yaml dice nc={nc}, pero names tiene {len(class_names)} clases. "
            f"Se usara names como fuente principal.",
            flush=True,
        )

    if not class_names:
        raise RuntimeError("No hay clases declaradas en data.yaml.")

    print("Clases leidas desde data.yaml:", flush=True)
    for class_id, class_name in sorted(class_names.items()):
        print(f"  {class_id}: {class_name}", flush=True)

    return class_names


def encontrar_carpeta_dataset():
    posibles_images = [p for p in DATASET_EXTRAIDO_DIR.rglob("images") if p.is_dir()]
    posibles_labels = [p for p in DATASET_EXTRAIDO_DIR.rglob("labels") if p.is_dir()]

    if not posibles_images:
        raise RuntimeError("No se encontro carpeta images dentro del ZIP.")

    if not posibles_labels:
        raise RuntimeError("No se encontro carpeta labels dentro del ZIP.")

    images_dir = posibles_images[0]
    labels_dir = posibles_labels[0]

    return images_dir, labels_dir


def buscar_imagenes(images_dir):
    extensiones = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return sorted([p for p in images_dir.rglob("*") if p.suffix.lower() in extensiones])


def crear_indice_etiquetas(labels_dir):
    indice = {}

    for txt in labels_dir.rglob("*.txt"):
        indice[txt.stem] = txt

    return indice


def buscar_label_correspondiente(ruta_imagen, indice_etiquetas):
    return indice_etiquetas.get(ruta_imagen.stem)


def leer_label_yolo(ruta_label, class_names):
    cajas = []
    errores = []

    clases_validas = set(class_names.keys())

    texto = ruta_label.read_text(encoding="utf-8", errors="ignore")

    for numero_linea, linea in enumerate(texto.splitlines(), start=1):
        linea = linea.strip()

        if not linea:
            continue

        partes = linea.split()

        if len(partes) < 5:
            errores.append(f"{ruta_label.name}:{numero_linea} linea incompleta")
            continue

        try:
            class_id = int(partes[0])
            x_center = float(partes[1])
            y_center = float(partes[2])
            width = float(partes[3])
            height = float(partes[4])
        except ValueError:
            errores.append(f"{ruta_label.name}:{numero_linea} valores no numericos")
            continue

        if class_id not in clases_validas:
            errores.append(
                f"{ruta_label.name}:{numero_linea} clase {class_id} no esta declarada en data.yaml"
            )
            continue

        if width <= 0 or height <= 0:
            errores.append(f"{ruta_label.name}:{numero_linea} caja con ancho/alto invalido")
            continue

        if not (0 <= x_center <= 1 and 0 <= y_center <= 1):
            errores.append(f"{ruta_label.name}:{numero_linea} centro fuera de rango 0..1")
            continue

        xmin = max(0.0, x_center - width / 2)
        ymin = max(0.0, y_center - height / 2)
        xmax = min(1.0, x_center + width / 2)
        ymax = min(1.0, y_center + height / 2)

        if xmax <= xmin or ymax <= ymin:
            errores.append(f"{ruta_label.name}:{numero_linea} caja sin area util")
            continue

        cajas.append({
            "class_id": class_id,
            "class_name": class_names[class_id],
            "xmin": xmin,
            "ymin": ymin,
            "xmax": xmax,
            "ymax": ymax,
        })

    return cajas, errores


def validar_dataset(imagenes, indice_etiquetas, class_names):
    registros = []
    errores_dataset = []
    imagenes_sin_txt = 0
    imagenes_sin_cajas = 0
    imagenes_invalidas = 0
    contador_clases = Counter()

    for ruta_imagen in imagenes:
        ruta_label = buscar_label_correspondiente(ruta_imagen, indice_etiquetas)

        if ruta_label is None:
            imagenes_sin_txt += 1
            errores_dataset.append({
                "imagen": str(ruta_imagen),
                "error": "No se encontro TXT correspondiente",
            })
            continue

        cajas, errores_label = leer_label_yolo(ruta_label, class_names)

        for error in errores_label:
            errores_dataset.append({
                "imagen": str(ruta_imagen),
                "label": str(ruta_label),
                "error": error,
            })

        if not cajas:
            imagenes_sin_cajas += 1
            continue

        try:
            with Image.open(ruta_imagen) as imagen:
                imagen.verify()

            with Image.open(ruta_imagen) as imagen:
                ancho, alto = imagen.size

            if ancho <= 0 or alto <= 0:
                raise ValueError("Dimensiones no validas")

        except (UnidentifiedImageError, OSError, ValueError) as exc:
            imagenes_invalidas += 1
            errores_dataset.append({
                "imagen": str(ruta_imagen),
                "error": str(exc),
            })
            continue

        for caja in cajas:
            contador_clases[caja["class_id"]] += 1

        registros.append({
            "image_path": str(ruta_imagen),
            "label_path": str(ruta_label),
            "image_width": ancho,
            "image_height": alto,
            "boxes": cajas,
        })

    df = pd.DataFrame(registros)

    print("=" * 60)
    print("VALIDACION DEL DATASET")
    print("=" * 60)
    print(f"Imagenes validas:           {len(df)}", flush=True)
    print(f"Imagenes sin TXT:           {imagenes_sin_txt}", flush=True)
    print(f"Imagenes sin cajas validas: {imagenes_sin_cajas}", flush=True)
    print(f"Imagenes invalidas:         {imagenes_invalidas}", flush=True)
    print("Cajas por clase:", flush=True)

    for class_id, class_name in sorted(class_names.items()):
        print(f"  {class_id}: {class_name} -> {contador_clases[class_id]} cajas", flush=True)

    if errores_dataset:
        pd.DataFrame(errores_dataset).to_csv(
            RESULTADOS_DIR / "errores_dataset.csv",
            index=False,
            encoding="utf-8-sig",
        )

    if len(df) < 10:
        raise RuntimeError("Se requieren al menos 10 imagenes validas para dividir train, validation y test.")

    return df


def dividir_dataset(df):
    random.seed(SEED)

    indices = list(df.index)
    random.shuffle(indices)

    total = len(indices)
    n_train = int(total * TRAIN_RATIO)
    n_val = int(total * VAL_RATIO)

    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train + n_val]
    test_idx = indices[n_train + n_val:]

    df = df.copy()
    df["split"] = "UNASSIGNED"
    df.loc[train_idx, "split"] = "TRAIN"
    df.loc[val_idx, "split"] = "VALIDATION"
    df.loc[test_idx, "split"] = "TEST"

    print(f"Train:      {(df['split'] == 'TRAIN').sum()} imagenes", flush=True)
    print(f"Validation: {(df['split'] == 'VALIDATION').sum()} imagenes", flush=True)
    print(f"Test:       {(df['split'] == 'TEST').sum()} imagenes", flush=True)

    return df


def crear_csv_model_maker(df, class_names):
    filas = []

    for _, row in df.iterrows():
        split = row["split"]

        for caja in row["boxes"]:
            filas.append([
                split,
                row["image_path"],
                caja["class_name"],
                caja["xmin"],
                caja["ymin"],
                "",
                "",
                caja["xmax"],
                caja["ymax"],
                "",
                "",
            ])

    csv_df = pd.DataFrame(filas)
    csv_df.to_csv(CSV_PATH, index=False, header=False, encoding="utf-8")

    LABELS_PATH.write_text(
        "\n".join(class_names[class_id] for class_id in sorted(class_names.keys())) + "\n",
        encoding="utf-8",
    )

    print(f"CSV para Model Maker: {CSV_PATH}", flush=True)
    print(f"Bounding boxes totales: {len(filas)}", flush=True)

    return CSV_PATH


def entrenar_y_exportar(class_names):
    print("Cargando CSV en TensorFlow Lite Model Maker...", flush=True)

    datos = object_detector.DataLoader.from_csv(str(CSV_PATH))

    if datos is None:
        raise RuntimeError("Model Maker no pudo cargar el CSV. Revise annotations_model_maker.csv")

    train_data, validation_data, test_data = datos

    print(f"Objetos train: {len(train_data)}", flush=True)
    print(f"Objetos validation: {len(validation_data)}", flush=True)
    print(f"Objetos test: {len(test_data)}", flush=True)

    print("Seleccionando arquitectura EfficientDet-Lite1...", flush=True)
    spec = model_spec.get("efficientdet_lite1")

    print(f"Iniciando entrenamiento: {EPOCHS} epocas, batch size {BATCH_SIZE}.", flush=True)

    model = object_detector.create(
        train_data,
        model_spec=spec,
        validation_data=validation_data,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        train_whole_model=True,
        verbose=2,
    )

    print("Evaluando modelo...", flush=True)
    metricas = model.evaluate(test_data)
    print(metricas, flush=True)

    (RESULTADOS_DIR / "metricas_evaluacion.json").write_text(
        json.dumps(str(metricas), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print("Exportando modelo Float32...", flush=True)
    model.export(
        export_dir=str(EXPORT_DIR),
        tflite_filename="efficientdet_lite1_botellas_float32.tflite",
        label_filename="labels.txt",
        export_format=[ExportFormat.TFLITE, ExportFormat.LABEL],
    )

    print("Exportando modelo Float16...", flush=True)
    model.export(
        export_dir=str(EXPORT_DIR),
        tflite_filename="efficientdet_lite1_botellas_fp16.tflite",
        label_filename="labels.txt",
        quantization_config=QuantizationConfig.for_float16(),
        export_format=[ExportFormat.TFLITE, ExportFormat.LABEL],
    )

    shutil.copy2(LABELS_PATH, EXPORT_DIR / "labels.txt")

    paquete = Path("/content/efficientdet_lite1_botellas_android.zip")
    if paquete.exists():
        paquete.unlink()

    with zipfile.ZipFile(paquete, "w", zipfile.ZIP_DEFLATED) as z:
        for archivo in EXPORT_DIR.rglob("*"):
            if archivo.is_file():
                z.write(archivo, f"modelo_exportado/{archivo.name}")

        for archivo in DATASET_PREPARADO_DIR.rglob("*"):
            if archivo.is_file():
                z.write(archivo, f"dataset_preparado/{archivo.name}")

        for archivo in RESULTADOS_DIR.rglob("*"):
            if archivo.is_file():
                z.write(archivo, f"resultados/{archivo.name}")

    print(f"Paquete final creado: {paquete}", flush=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--zip", required=True, help="Ruta al ZIP YOLO")
    args = parser.parse_args()

    zip_path = Path(args.zip)

    if not zip_path.exists():
        raise FileNotFoundError(zip_path)

    print("=" * 70)
    print("PASO 1/8: Limpiar carpetas de ejecuciones anteriores")
    print("=" * 70)
    limpiar_carpetas()

    print("\n" + "=" * 70)
    print("PASO 2/8: Extraer el ZIP del dataset")
    print("=" * 70)
    extraer_zip(zip_path)

    print("\n" + "=" * 70)
    print("PASO 3/8: Buscar imagenes, etiquetas YOLO y data.yaml")
    print("=" * 70)
    yaml_path = encontrar_data_yaml()
    class_names = cargar_clases_desde_yaml(yaml_path)
    images_dir, labels_dir = encontrar_carpeta_dataset()
    imagenes = buscar_imagenes(images_dir)
    indice_etiquetas = crear_indice_etiquetas(labels_dir)

    print(f"Imagenes encontradas: {len(imagenes)}", flush=True)
    print(f"Etiquetas TXT encontradas: {len(indice_etiquetas)}", flush=True)

    print("\n" + "=" * 70)
    print("PASO 4/8: Validar imagenes y bounding boxes")
    print("=" * 70)
    df = validar_dataset(imagenes, indice_etiquetas, class_names)

    print("\n" + "=" * 70)
    print("PASO 5/8: Dividir dataset en train, validation y test")
    print("=" * 70)
    df = dividir_dataset(df)

    print("\n" + "=" * 70)
    print("PASO 6/8: Crear CSV y archivos auxiliares para Model Maker")
    print("=" * 70)
    crear_csv_model_maker(df, class_names)

    print("\n" + "=" * 70)
    print("PASO 7/8: Entrenar, evaluar y exportar modelos TFLite")
    print("=" * 70)
    entrenar_y_exportar(class_names)

    print("\n" + "=" * 70)
    print("PASO 8/8: Finalizado")
    print("=" * 70)
    print("Entrenamiento y exportacion completados correctamente.", flush=True)


if __name__ == "__main__":
    main()
'''

script_path.write_text(codigo, encoding="utf-8")

print(f"Script creado correctamente: {script_path}")
print("Listo. Ahora ejecuta la Celda 4.")

Script creado correctamente: /content/train_efficientdet_lite1.py
Listo. Ahora ejecuta la Celda 4.


## Celda 4. Entrenar y exportar modelos

Esta celda ejecuta el entrenamiento dentro del entorno Python 3.9. Al finalizar genera `/content/efficientdet_lite1_botellas_android.zip`.


In [9]:
# ============================================================
# PARCHEAR SCRIPT PARA ENTRENAR TODAS LAS CLASES DEL data.yaml
# ============================================================

from pathlib import Path
import re

script_path = Path("/content/train_efficientdet_lite1.py")

if not script_path.exists():
    raise FileNotFoundError(
        "No existe /content/train_efficientdet_lite1.py. "
        "Ejecute primero la celda que genera el script de entrenamiento."
    )

codigo = script_path.read_text(encoding="utf-8")

reemplazos = [
    # Casos tipicos donde el script filtra solo clase 0
    ("if class_id != 0:", "if class_id not in class_names:"),
    ("if cls_id != 0:", "if cls_id not in class_names:"),
    ("if clase_id != 0:", "if clase_id not in class_names:"),
    ("if class_id == 0:", "if class_id in class_names:"),
    ("if cls_id == 0:", "if cls_id in class_names:"),
    ("if clase_id == 0:", "if clase_id in class_names:"),
]

aplicados = 0

for viejo, nuevo in reemplazos:
    if viejo in codigo:
        codigo = codigo.replace(viejo, nuevo)
        aplicados += 1

# Si el script fuerza el nombre de clase a una sola etiqueta, lo corregimos
codigo = codigo.replace('class_name = "botella"', 'class_name = class_names[class_id]')
codigo = codigo.replace("class_name = 'botella'", "class_name = class_names[class_id]")
codigo = codigo.replace('class_name = "producto"', 'class_name = class_names[class_id]')
codigo = codigo.replace("class_name = 'producto'", "class_name = class_names[class_id]")

# Variante con cls_id
codigo = codigo.replace('class_name = class_names[0]', 'class_name = class_names[class_id]')
codigo = codigo.replace('label = class_names[0]', 'label = class_names[class_id]')

script_path.write_text(codigo, encoding="utf-8")

print("Script revisado:", script_path)

if aplicados == 0:
    print("No encontre un filtro simple tipo class_id != 0.")
    print("Si sigue usando 178 imagenes, pegaremos aqui el contenido de la celda 3 y lo ajusto exacto.")
else:
    print(f"Filtros modificados: {aplicados}")

print("Listo. Ahora ejecuta la celda 4.")

Script revisado: /content/train_efficientdet_lite1.py
No encontre un filtro simple tipo class_id != 0.
Si sigue usando 178 imagenes, pegaremos aqui el contenido de la celda 3 y lo ajusto exacto.
Listo. Ahora ejecuta la celda 4.


In [ ]:
# ============================================================
# EJECUTAR ENTRENAMIENTO Y MOSTRAR PROGRESO EN TIEMPO REAL
# ============================================================

import subprocess
import sys
from pathlib import Path

python_entorno = Path("/content/miniconda/envs/tflite_mm39/bin/python")
script_entrenamiento = Path("/content/train_efficientdet_lite1.py")
dataset_zip = Path("/content/dataset_botellas.zip")

if not python_entorno.exists():
    raise FileNotFoundError(
        "No existe el Python del entorno tflite_mm39. "
        "Ejecute primero la celda 1."
    )

if not script_entrenamiento.exists():
    raise FileNotFoundError(
        "No existe /content/train_efficientdet_lite1.py. "
        "Ejecute primero la celda 3 para generar el script."
    )

if not dataset_zip.exists():
    raise FileNotFoundError(
        "No existe /content/dataset_botellas.zip. "
        "Ejecute primero la celda 2 y suba el ZIP del dataset."
    )

comando = [
    str(python_entorno),
    "-u",
    str(script_entrenamiento),
    "--zip",
    str(dataset_zip),
]

print("=" * 70, flush=True)
print("INICIANDO CELDA 4: entrenamiento y exportaci?n", flush=True)
print("La salida se mostrar? en tiempo real debajo de esta celda.", flush=True)
print("Comando:", " ".join(comando), flush=True)
print("=" * 70, flush=True)

proceso = subprocess.Popen(
    comando,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for linea in proceso.stdout:
    print(linea, end="", flush=True)

codigo_salida = proceso.wait()

if codigo_salida != 0:
    raise subprocess.CalledProcessError(codigo_salida, comando)

print("\n" + "=" * 70, flush=True)
print("CELDA 4 FINALIZADA CORRECTAMENTE", flush=True)
print("Ahora puede ejecutar la celda 5 para descargar el ZIP final.", flush=True)
print("=" * 70, flush=True)


INICIANDO CELDA 4: entrenamiento y exportaci?n
La salida se mostrar? en tiempo real debajo de esta celda.
Comando: /content/miniconda/envs/tflite_mm39/bin/python -u /content/train_efficientdet_lite1.py --zip /content/dataset_botellas.zip
2026-08-03 19:27:43.026767: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/lib64-nvidia
2026-08-03 19:27:43.026825: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
/content/miniconda/envs/tflite_mm39/lib/python3.9/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 

## Celda 5. Probar el modelo con una imagen

Ejecute esta celda después de la celda 4. Permite subir una imagen, correr el modelo TFLite Float16 exportado y ver la foto con las detecciones dibujadas.


In [ ]:
# ============================================================
# SUBIR IMAGEN Y MOSTRAR PREDICCI?N DEL MODELO
# ============================================================

from pathlib import Path

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw, ImageFont
from google.colab import files

MODELO_PRUEBA = Path("/content/efficientdet_lite1_botellas/modelo_exportado/efficientdet_lite1_botellas_fp16.tflite")
LABELS_PRUEBA = Path("/content/efficientdet_lite1_botellas/dataset_preparado/labels.txt")

UMBRAL_CONFIANZA = 0.35
MAX_DETECCIONES = 20

if not MODELO_PRUEBA.exists():
    raise FileNotFoundError(
        "No se encontr? el modelo TFLite. "
        "Ejecute primero la celda 4 para entrenar y exportar el modelo."
    )

if LABELS_PRUEBA.exists():
    etiquetas = [
        linea.strip()
        for linea in LABELS_PRUEBA.read_text(encoding="utf-8").splitlines()
        if linea.strip()
    ]
else:
    etiquetas = ["botellas"]

print("Seleccione una imagen para probar el detector.")
archivos_subidos = files.upload()

if not archivos_subidos:
    raise RuntimeError("No se seleccion? ninguna imagen.")

nombre_imagen = next(iter(archivos_subidos))
ruta_imagen = Path("/content") / nombre_imagen

imagen_original = Image.open(ruta_imagen).convert("RGB")
ancho_original, alto_original = imagen_original.size

interpreter = tf.lite.Interpreter(model_path=str(MODELO_PRUEBA))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

entrada = input_details[0]
_, alto_entrada, ancho_entrada, canales = entrada["shape"]

imagen_redimensionada = imagen_original.resize((ancho_entrada, alto_entrada))
input_data = np.expand_dims(np.asarray(imagen_redimensionada), axis=0)

if entrada["dtype"] == np.float32:
    input_data = input_data.astype(np.float32)
else:
    input_data = input_data.astype(entrada["dtype"])

interpreter.set_tensor(entrada["index"], input_data)
interpreter.invoke()

salidas = {
    detalle["name"].lower(): interpreter.get_tensor(detalle["index"])
    for detalle in output_details
}

boxes = None
classes = None
scores = None
num_detections = None

for nombre, valor in salidas.items():
    valor_sin_batch = np.squeeze(valor)

    if "box" in nombre and valor_sin_batch.ndim == 2 and valor_sin_batch.shape[-1] == 4:
        boxes = valor_sin_batch
    elif "score" in nombre and valor_sin_batch.ndim == 1:
        scores = valor_sin_batch
    elif "class" in nombre and valor_sin_batch.ndim == 1:
        classes = valor_sin_batch
    elif "num" in nombre:
        num_detections = int(np.squeeze(valor))

# Respaldo para modelos que no tienen nombres de salida claros.
if boxes is None:
    for valor in salidas.values():
        valor_sin_batch = np.squeeze(valor)
        if valor_sin_batch.ndim == 2 and valor_sin_batch.shape[-1] == 4:
            boxes = valor_sin_batch
            break

if scores is None or classes is None:
    vectores = []
    for valor in salidas.values():
        valor_sin_batch = np.squeeze(valor)
        if valor_sin_batch.ndim == 1 and valor_sin_batch.size > 1:
            vectores.append(valor_sin_batch)

    if scores is None and vectores:
        scores = max(vectores, key=lambda vector: float(np.nanmax(vector)))

    if classes is None:
        for vector in vectores:
            if scores is not None and vector is scores:
                continue
            classes = vector
            break

if boxes is None or scores is None:
    print("Detalles de salida del modelo:")
    for detalle in output_details:
        print(detalle["name"], detalle["shape"], detalle["dtype"])
    raise RuntimeError("No se pudieron interpretar las salidas del modelo TFLite.")

cantidad = num_detections or min(len(boxes), len(scores), MAX_DETECCIONES)
cantidad = min(cantidad, len(boxes), len(scores), MAX_DETECCIONES)

imagen_resultado = imagen_original.copy()
dibujo = ImageDraw.Draw(imagen_resultado)

try:
    fuente = ImageFont.truetype("DejaVuSans.ttf", 18)
except OSError:
    fuente = ImageFont.load_default()

detecciones = []

for i in range(cantidad):
    score = float(scores[i])
    if score < UMBRAL_CONFIANZA:
        continue

    ymin, xmin, ymax, xmax = [float(v) for v in boxes[i]]

    # Algunos modelos pueden devolver coordenadas en p?xeles; si vienen normalizadas,
    # las escalamos al tama?o original.
    if max(abs(xmin), abs(ymin), abs(xmax), abs(ymax)) <= 1.5:
        x1 = int(xmin * ancho_original)
        y1 = int(ymin * alto_original)
        x2 = int(xmax * ancho_original)
        y2 = int(ymax * alto_original)
    else:
        x1 = int(xmin)
        y1 = int(ymin)
        x2 = int(xmax)
        y2 = int(ymax)

    x1 = max(0, min(x1, ancho_original - 1))
    y1 = max(0, min(y1, alto_original - 1))
    x2 = max(0, min(x2, ancho_original - 1))
    y2 = max(0, min(y2, alto_original - 1))

    if x2 <= x1 or y2 <= y1:
        continue

    class_id = int(classes[i]) if classes is not None and i < len(classes) else 0
    if class_id >= len(etiquetas) and class_id - 1 >= 0 and class_id - 1 < len(etiquetas):
        class_id -= 1

    etiqueta = etiquetas[class_id] if 0 <= class_id < len(etiquetas) else f"clase_{class_id}"
    texto = f"{etiqueta}: {score:.2f}"

    dibujo.rectangle([x1, y1, x2, y2], outline="lime", width=4)

    bbox_texto = dibujo.textbbox((x1, y1), texto, font=fuente)
    tx1, ty1, tx2, ty2 = bbox_texto
    alto_texto = ty2 - ty1

    fondo_y1 = max(0, y1 - alto_texto - 8)
    dibujo.rectangle([x1, fondo_y1, x1 + (tx2 - tx1) + 8, y1], fill="lime")
    dibujo.text((x1 + 4, fondo_y1 + 3), texto, fill="black", font=fuente)

    detecciones.append({
        "clase": etiqueta,
        "confianza": score,
        "bbox": [x1, y1, x2, y2],
    })

print("=" * 60)
print("RESULTADO DE LA PREDICCI?N")
print("=" * 60)
print(f"Imagen: {nombre_imagen}")
print(f"Detecciones sobre umbral {UMBRAL_CONFIANZA}: {len(detecciones)}")

for numero, deteccion in enumerate(detecciones, start=1):
    print(
        f"{numero}. {deteccion['clase']} "
        f"confianza={deteccion['confianza']:.3f} "
        f"bbox={deteccion['bbox']}"
    )

plt.figure(figsize=(10, 10))
plt.imshow(imagen_resultado)
plt.axis("off")
plt.show()


FileNotFoundError: No se encontr? el modelo TFLite. Ejecute primero la celda 4 para entrenar y exportar el modelo.

## Celda 6. Descargar resultado


In [ ]:
# ============================================================
# DESCARGAR PAQUETE FINAL
# ============================================================

from pathlib import Path
from google.colab import files

zip_final = Path("/content/efficientdet_lite1_botellas_android.zip")

if not zip_final.exists():
    raise FileNotFoundError("Todav?a no existe el ZIP final. Ejecute primero la celda de entrenamiento.")

print(f"Descargando: {zip_final.name}")
files.download(str(zip_final))


## Uso básico en Android

Coloque el modelo elegido en:

```text
app/src/main/assets/efficientdet_lite1_botellas_fp16.tflite
```

Dependencias sugeridas:

```gradle
dependencies {
    implementation "org.tensorflow:tensorflow-lite-task-vision"
    implementation "org.tensorflow:tensorflow-lite-gpu-delegate-plugin"
}
```

Inicializaci?n en Kotlin:

```kotlin
import org.tensorflow.lite.task.core.BaseOptions
import org.tensorflow.lite.task.vision.detector.ObjectDetector

val baseOptions = BaseOptions.builder()
    .useGpu()
    .build()

val options = ObjectDetector.ObjectDetectorOptions.builder()
    .setBaseOptions(baseOptions)
    .setScoreThreshold(0.35f)
    .setMaxResults(20)
    .build()

val detector = ObjectDetector.createFromFileAndOptions(
    context,
    "efficientdet_lite1_botellas_fp16.tflite",
    options
)
```

Si el GPU Delegate da problemas en un tel?fono, pruebe el modelo INT8 y quite `.useGpu()`.
